# Milestone 2: Data Quality dan Profiling

Lanjutkan dari hasil `01_data_profiling.ipynb`. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data dasar. Sesuaikan nama kolom dan aturan validasi dengan dataset masing-masing.

In [43]:
from pathlib import Path
import duckdb
import polars as pl

CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

print("Project root:", PROJECT_ROOT)

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "globalLandTemperaturesByCity.csv"
)

OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "dataset_bersih.parquet"
)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset CSV tidak ditemukan:\n{DATA_PATH}"
    )

print("Dataset ditemukan:", DATA_PATH)

df = pl.read_csv(
    DATA_PATH,
    infer_schema_length=10000,
    ignore_errors=True,
    truncate_ragged_lines=True
)

print("\nDataset berhasil dibaca!")
print(f"Ukuran awal: {df.height:,} baris x {df.width} kolom")
display(df.head())

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.write_parquet(OUTPUT_PATH)

print("\nDataset berhasil disimpan ke Parquet!")
print("Lokasi:", OUTPUT_PATH)

Project root: C:\Users\NISA\tugas1-abd-2026
Dataset ditemukan: C:\Users\NISA\tugas1-abd-2026\data\raw\globalLandTemperaturesByCity.csv

Dataset berhasil dibaca!
Ukuran awal: 457,044 baris x 7 kolom


1951-01-01,-0.311,0.486,Århus,Denmark,57.05N,10.33E
str,f64,f64,str,str,str,str
"""1951-02-01""",0.62901,0.506,"""Århus""","""Denmark""","""57.05N""","""10.33E"""
"""1951-03-01""",0.034,0.392,"""Århus""","""Denmark""","""57.05N""","""10.33E"""
"""1951-04-01""",5.555,0.189,"""Århus""","""Denmark""","""57.05N""","""10.33E"""
"""1951-05-01""",10.458,0.282,"""Århus""","""Denmark""","""57.05N""","""10.33E"""
"""1951-06-01""",14.99402,0.622,"""Århus""","""Denmark""","""57.05N""","""10.33E"""



Dataset berhasil disimpan ke Parquet!
Lokasi: C:\Users\NISA\tugas1-abd-2026\data\processed\dataset_bersih.parquet


In [44]:
df = pl.read_parquet(OUTPUT_PATH)

print(f"Ukuran data: {df.height:,} baris x {df.width} kolom")

display(df.head())

Ukuran data: 457,044 baris x 7 kolom


1951-01-01,-0.311,0.486,Århus,Denmark,57.05N,10.33E
str,f64,f64,str,str,str,str
"""1951-02-01""",0.62901,0.506,"""Århus""","""Denmark""","""57.05N""","""10.33E"""
"""1951-03-01""",0.034,0.392,"""Århus""","""Denmark""","""57.05N""","""10.33E"""
"""1951-04-01""",5.555,0.189,"""Århus""","""Denmark""","""57.05N""","""10.33E"""
"""1951-05-01""",10.458,0.282,"""Århus""","""Denmark""","""57.05N""","""10.33E"""
"""1951-06-01""",14.99402,0.622,"""Århus""","""Denmark""","""57.05N""","""10.33E"""


## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom (tipe data, null%, min/max, distinct count) dalam satu perintah.

In [45]:
import duckdb
import polars as pl

connection = duckdb.connect()

connection.register("dataset", df)

result = connection.execute(
    "SUMMARIZE SELECT * FROM dataset"
)

hasil = result.fetchall()
nama_kolom = [desc[0] for desc in result.description]

profil = pl.DataFrame(
    hasil,
    schema=nama_kolom,
    orient="row"
)

display(profil)

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[38,2]"
"""1951-01-01""","""VARCHAR""","""1951-01-01""","""1999-12-01""",301,null,null,null,null,null,457044,0.00
"""-0.311""","""DOUBLE""","""-32.086999999999996""","""38.389""",56942,"""18.477330864151277""","""9.705829442306598""","""13.123774059950552""","""20.836290503647223""","""25.999349571194756""",457044,0.00
"""0.486""","""DOUBLE""","""0.04""","""4.1419999999999995""",1898,"""0.36692528357444837""","""0.19197749924979332""","""0.24204748634448287""","""0.3236794660034422""","""0.4393551964130621""",457044,0.00
"""Århus""","""VARCHAR""","""A Coruña""","""Ürümqi""",1565,null,null,null,null,null,457044,0.00
"""Denmark""","""VARCHAR""","""Afghanistan""","""Zimbabwe""",124,null,null,null,null,null,457044,0.00
"""57.05N""","""VARCHAR""","""0.80N""","""8.84S""",68,null,null,null,null,null,457044,0.00
"""10.33E""","""VARCHAR""","""0.00W""","""99.91E""",741,null,null,null,null,null,457044,0.00


## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom sebelum memutuskan strategi penanganan.

In [46]:
jumlah_baris = len(df)
null_summary = (
    df.null_count()
    .transpose(include_header=True, header_name='kolom', column_names=['jumlah_null'])
    .with_columns(
        (pl.col('jumlah_null') / jumlah_baris * 100).round(2).alias('persen_null')
    )
    .filter(pl.col('jumlah_null') > 0)
    .sort('jumlah_null', descending=True)
)
null_summary

kolom,jumlah_null,persen_null
str,u32,f64
"""57.05N""",1,0.0
"""10.33E""",1,0.0


## 3. Penanganan Missing Values

Ganti `kolom_numerik` dan `kolom_kategorik` dengan nama kolom asli pada dataset. Pilih strategi sesuai jenis data: median/mean untuk numerik, nilai default atau modus untuk kategorik.

In [47]:
# TODO: sesuaikan nama kolom dengan dataset Anda
df_missing_handled = df.with_columns([
    # Contoh: isi kolom numerik dengan median
    # pl.col('kolom_numerik').fill_null(pl.col('kolom_numerik').median()),

    # Contoh: isi kolom kategorik dengan nilai default
    # pl.col('kolom_kategorik').fill_null('tidak diketahui'),
])

print(f'Null sebelum: {df.null_count().sum_horizontal().sum():,}')
print(f'Null sesudah: {df_missing_handled.null_count().sum_horizontal().sum():,}')

Null sebelum: 2
Null sesudah: 2


## 4. Deteksi dan Penanganan Outlier

Gunakan metode IQR sebagai titik awal, lalu kombinasikan dengan batas bisnis (domain knowledge) yang relevan dengan dataset Anda.

In [48]:
from pathlib import Path

with open(DATA_PATH, "r", encoding="utf-8", errors="replace") as file:
    for i in range(5):
        print(f"Baris {i+1}: {file.readline().strip()}")

Baris 1: 1951-01-01,-0.311,0.486,Århus,Denmark,57.05N,10.33E
Baris 2: 1951-02-01,0.62901,0.506,Århus,Denmark,57.05N,10.33E
Baris 3: 1951-03-01,0.033999999999999815,0.392,Århus,Denmark,57.05N,10.33E
Baris 4: 1951-04-01,5.555,0.18899999999999997,Århus,Denmark,57.05N,10.33E
Baris 5: 1951-05-01,10.458,0.282,Århus,Denmark,57.05N,10.33E


In [49]:
import polars as pl

nama_kolom = [
    "dt",
    "AverageTemperature",
    "AverageTemperatureUncertainty",
    "City",
    "Country",
    "Latitude",
    "Longitude"
]

df = pl.read_csv(
    DATA_PATH,
    has_header=False,
    new_columns=nama_kolom,
    infer_schema_length=10000,
    ignore_errors=True,
    truncate_ragged_lines=True
)

print("Ukuran data:", df.shape)
print("Kolom:", df.columns)
display(df.head())

Ukuran data: (457045, 7)
Kolom: ['dt', 'AverageTemperature', 'AverageTemperatureUncertainty', 'City', 'Country', 'Latitude', 'Longitude']


dt,AverageTemperature,AverageTemperatureUncertainty,City,Country,Latitude,Longitude
str,f64,f64,str,str,str,str
"""1951-01-01""",-0.311,0.486,"""Århus""","""Denmark""","""57.05N""","""10.33E"""
"""1951-02-01""",0.62901,0.506,"""Århus""","""Denmark""","""57.05N""","""10.33E"""
"""1951-03-01""",0.034,0.392,"""Århus""","""Denmark""","""57.05N""","""10.33E"""
"""1951-04-01""",5.555,0.189,"""Århus""","""Denmark""","""57.05N""","""10.33E"""
"""1951-05-01""",10.458,0.282,"""Århus""","""Denmark""","""57.05N""","""10.33E"""


## 5. Deduplikasi

Tentukan kolom kunci bisnis (misalnya ID unik) sebelum menghapus duplikat — jangan deduplikasi berdasarkan seluruh baris.

In [50]:
# Periksa nama kolom dataset
print(df.columns)

# Periksa beberapa baris data
print(df.head())

# Periksa jumlah baris
print(f"Jumlah baris: {len(df):,}")

['dt', 'AverageTemperature', 'AverageTemperatureUncertainty', 'City', 'Country', 'Latitude', 'Longitude']
shape: (5, 7)
┌────────────┬────────────────────┬───────────────────────┬───────┬─────────┬──────────┬───────────┐
│ dt         ┆ AverageTemperature ┆ AverageTemperatureUnc ┆ City  ┆ Country ┆ Latitude ┆ Longitude │
│ ---        ┆ ---                ┆ ertainty              ┆ ---   ┆ ---     ┆ ---      ┆ ---       │
│ str        ┆ f64                ┆ ---                   ┆ str   ┆ str     ┆ str      ┆ str       │
│            ┆                    ┆ f64                   ┆       ┆         ┆          ┆           │
╞════════════╪════════════════════╪═══════════════════════╪═══════╪═════════╪══════════╪═══════════╡
│ 1951-01-01 ┆ -0.311             ┆ 0.486                 ┆ Århus ┆ Denmark ┆ 57.05N   ┆ 10.33E    │
│ 1951-02-01 ┆ 0.62901            ┆ 0.506                 ┆ Århus ┆ Denmark ┆ 57.05N   ┆ 10.33E    │
│ 1951-03-01 ┆ 0.034              ┆ 0.392                 ┆ Århus ┆ Denm

In [51]:
print("Nama kolom pada dataset:")
for i, kolom in enumerate(df_outlier_handled.columns):
    print(f"{i+1}. {kolom}")

Nama kolom pada dataset:
1. dt
2. AverageTemperature
3. AverageTemperatureUncertainty
4. City
5. Country
6. Latitude
7. Longitude


In [52]:
print(df_outlier_handled.schema)
print(df_outlier_handled.head(10))

Schema({'dt': String, 'AverageTemperature': Float64, 'AverageTemperatureUncertainty': Float64, 'City': String, 'Country': String, 'Latitude': String, 'Longitude': String})
shape: (10, 7)
┌────────────┬────────────────────┬───────────────────────┬───────┬─────────┬──────────┬───────────┐
│ dt         ┆ AverageTemperature ┆ AverageTemperatureUnc ┆ City  ┆ Country ┆ Latitude ┆ Longitude │
│ ---        ┆ ---                ┆ ertainty              ┆ ---   ┆ ---     ┆ ---      ┆ ---       │
│ str        ┆ f64                ┆ ---                   ┆ str   ┆ str     ┆ str      ┆ str       │
│            ┆                    ┆ f64                   ┆       ┆         ┆          ┆           │
╞════════════╪════════════════════╪═══════════════════════╪═══════╪═════════╪══════════╪═══════════╡
│ 1951-01-01 ┆ -0.311             ┆ 0.486                 ┆ Århus ┆ Denmark ┆ 57.05N   ┆ 10.33E    │
│ 1951-02-01 ┆ 0.62901            ┆ 0.506                 ┆ Århus ┆ Denmark ┆ 57.05N   ┆ 10.33E    │
│ 195

In [53]:
KOLOM_KUNCI = [
    'dt',
    'City',
    'Country',
    'Latitude',
    'Longitude'
]

total_baris = df_outlier_handled.height
baris_unik = df_outlier_handled.select(
    KOLOM_KUNCI
).unique().height
jumlah_duplikat = total_baris - baris_unik

print("===== SEBELUM DEDUPLIKASI =====")
print(f"Total baris       : {total_baris:,}")
print(f"Baris unik        : {baris_unik:,}")
print(f"Jumlah duplikat   : {jumlah_duplikat:,}")

df_dedup = df_outlier_handled.unique(
    subset=KOLOM_KUNCI,
    keep='first',
    maintain_order=True
)

print("\n===== HASIL DEDUPLIKASI =====")
print(f"Total baris awal  : {total_baris:,}")
print(f"Total baris akhir : {df_dedup.height:,}")
print(f"Data dihapus      : {total_baris - df_dedup.height:,}")

print("\n===== DATA SETELAH DEDUPLIKASI =====")
print(df_dedup.head())


===== SEBELUM DEDUPLIKASI =====
Total baris       : 457,045
Baris unik        : 457,045
Jumlah duplikat   : 0

===== HASIL DEDUPLIKASI =====
Total baris awal  : 457,045
Total baris akhir : 457,045
Data dihapus      : 0

===== DATA SETELAH DEDUPLIKASI =====
shape: (5, 7)
┌────────────┬────────────────────┬───────────────────────┬───────┬─────────┬──────────┬───────────┐
│ dt         ┆ AverageTemperature ┆ AverageTemperatureUnc ┆ City  ┆ Country ┆ Latitude ┆ Longitude │
│ ---        ┆ ---                ┆ ertainty              ┆ ---   ┆ ---     ┆ ---      ┆ ---       │
│ str        ┆ f64                ┆ ---                   ┆ str   ┆ str     ┆ str      ┆ str       │
│            ┆                    ┆ f64                   ┆       ┆         ┆          ┆           │
╞════════════╪════════════════════╪═══════════════════════╪═══════╪═════════╪══════════╪═══════════╡
│ 1951-01-01 ┆ -0.311             ┆ 0.486                 ┆ Århus ┆ Denmark ┆ 57.05N   ┆ 10.33E    │
│ 1951-02-01 ┆ 0.62901

## 6. Validasi Data Dasar

Tulis beberapa aturan validasi sederhana untuk memastikan data bersih memenuhi batas yang masuk akal. Gantikan contoh aturan di bawah dengan aturan yang relevan dengan dataset Anda.

In [54]:
import duckdb
import polars as pl

con_validasi = duckdb.connect()
con_validasi.register('dataset_bersih', df_dedup)

KOLOM_NUMERIK = 'AverageTemperature'
batas_bawah = -90
batas_atas = 60

hasil_query = con_validasi.execute(f"""
    SELECT
        COUNT(*) AS total_baris,

        COUNT(*) FILTER (
            WHERE "{KOLOM_NUMERIK}" IS NULL
        ) AS pelanggaran_null,

        COUNT(*) FILTER (
            WHERE "{KOLOM_NUMERIK}" < {batas_bawah}
               OR "{KOLOM_NUMERIK}" > {batas_atas}
        ) AS pelanggaran_batas

    FROM dataset_bersih
""").fetchall()

nama_kolom = [
    deskripsi[0]
    for deskripsi in con_validasi.description
]

validasi = pl.DataFrame(
    hasil_query,
    schema=nama_kolom,
    orient='row'
)

print("===== HASIL VALIDASI DATA =====")
print(validasi)

# Menutup koneksi DuckDB
con_validasi.close()


===== HASIL VALIDASI DATA =====
shape: (1, 3)
┌─────────────┬──────────────────┬───────────────────┐
│ total_baris ┆ pelanggaran_null ┆ pelanggaran_batas │
│ ---         ┆ ---              ┆ ---               │
│ i64         ┆ i64              ┆ i64               │
╞═════════════╪══════════════════╪═══════════════════╡
│ 457045      ┆ 0                ┆ 0                 │
└─────────────┴──────────────────┴───────────────────┘


## 7. Simpan Data Bersih

Simpan hasil ke `data/processed/` agar dapat dipakai pada Milestone 3 tanpa mengulang proses cleaning.

In [56]:
# Simpan data bersih
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df_dedup.write_parquet(OUTPUT_PATH)

print(f'Baris sebelum cleaning: {len(df):,}')
print(f'Baris sesudah cleaning: {len(df_dedup):,}')
print(f'Data bersih tersimpan di: {OUTPUT_PATH}')

Baris sebelum cleaning: 457,045
Baris sesudah cleaning: 457,045
Data bersih tersimpan di: C:\Users\NISA\tugas1-abd-2026\data\processed\dataset_bersih.parquet


## Catatan Milestone 2

Dokumentasikan di sini: dimensi kualitas data yang paling bermasalah, strategi yang dipilih untuk missing values dan outlier, jumlah baris yang berubah, serta aturan validasi yang diterapkan.